# GlaucomAI
### A clearer view. An earlier signal.
**Train on Kaggle data → evaluate on held-out images → try the model here.**

[GitHub source](https://github.com/Cxsmxnaut/GlaucomAI) ·
[Kaggle dataset](https://www.kaggle.com/datasets/deathtrooper/glaucoma-dataset-eyepacs-airogs-light-v2/versions/4)

This notebook includes all project source code. You do not need to clone a repository or run a web server.
In Colab, choose **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
GPU availability depends on your Colab account. Download and training time depend on the assigned runtime.

**Research prototype, not a diagnosis.** The classes are **non-referable** and **referable glaucoma**.
A non-referable result does not mean the eye is healthy. No clinical performance is promised.

## 1. Install the notebook tools
Keep Colab's preinstalled GPU-enabled PyTorch and Torchvision. This installs dataset access,
the upload controls, and plotting tools. No API keys belong in notebook cells or GitHub.

In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "kagglehub>=0.3.13,<1", "ipywidgets>=7.7,<9", "matplotlib>=3.9,<4", "Pillow>=11,<13"], check=True)
import torch
import torchvision
print("PyTorch:", torch.__version__, "| Torchvision:", torchvision.__version__)
print("Training device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (slower)")

## 2. Load the project
The next cell writes the same Python files tested in GitHub into this runtime's working directory.
The notebook is generated from those files, so the notebook and repository stay in sync.

In [ ]:
# @title Initialize the GlaucomAI project
import json
import os
from pathlib import Path
PROJECT = Path('/content/GlaucomAI') if Path('/content').exists() else Path.cwd() / '.glaucomai-notebook'
PROJECT.mkdir(parents=True, exist_ok=True)
SOURCES = json.loads('{"evaluate.py": "\\"\\"\\"Evaluate a locked model on held-out images; report real CPU latency.\\"\\"\\"\\n\\nimport argparse\\nimport hashlib\\nimport json\\nfrom pathlib import Path\\nimport statistics\\nimport time\\n\\nimport torch\\n\\nfrom glaucoma.data import FundusDataset, check_split_overlap\\nfrom glaucoma.metrics import binary_metrics\\nfrom glaucoma.pipeline import load_model\\n\\n\\ndef main():\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\\"--data\\", type=Path, default=Path(\\"data/prepared\\"))\\n    parser.add_argument(\\"--model\\", type=Path, default=Path(\\"models/best_model.pth\\"))\\n    parser.add_argument(\\"--output\\", type=Path, default=Path(\\"reports/evaluation.json\\"))\\n    args = parser.parse_args()\\n    torch.set_num_threads(2)\\n    train, val, test = [FundusDataset(args.data / split) for split in (\\"train\\", \\"val\\", \\"test\\")]\\n    check_split_overlap(train, val, test)\\n    loaded = load_model(args.model)\\n    labels, scores, latencies = [], [], []\\n    with torch.inference_mode():\\n        # Report cold model execution separately from warmed per-image pipeline.\\n        warmup_start = time.perf_counter()\\n        loaded.model(test[0][0].unsqueeze(0))\\n        warmup_ms = (time.perf_counter() - warmup_start) * 1000\\n        for index in range(len(test)):\\n            start = time.perf_counter()\\n            image, label = test[index]\\n            score = loaded.model(image.unsqueeze(0)).softmax(dim=1)[0, 1].item()\\n            latencies.append((time.perf_counter() - start) * 1000)\\n            labels.append(label)\\n            scores.append(score)\\n    metrics = binary_metrics(labels, scores, loaded.threshold)\\n    sorted_times = sorted(latencies)\\n    p95 = sorted_times[min(len(sorted_times) - 1, int(0.95 * len(sorted_times)))]\\n    report = {\\n        \\"split\\": \\"test\\", \\"threshold\\": loaded.threshold, \\"metrics\\": metrics,\\n        \\"checkpoint_sha256\\": hashlib.sha256(args.model.read_bytes()).hexdigest(),\\n        \\"warmup_ms\\": warmup_ms, \\"median_pipeline_ms\\": statistics.median(latencies),\\n        \\"p95_pipeline_ms\\": p95,\\n        \\"sensitivity_target_met\\": metrics[\\"sensitivity\\"] >= 0.88,\\n        \\"latency_target_met\\": p95 <= 2000,\\n        \\"note\\": \\"Latency includes file decoding, preprocessing and CPU forward pass, not browser transport or UI rendering. Patient separation must be verified by the dataset owner. Model scores are not calibrated clinical probabilities.\\",\\n    }\\n    args.output.parent.mkdir(parents=True, exist_ok=True)\\n    args.output.write_text(json.dumps(report, indent=2))\\n    print(json.dumps(report, indent=2))\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "glaucoma/__init__.py": "\\"\\"\\"GlaucomAI research prototype. No clinical validation is implied.\\"\\"\\"\\n", "glaucoma/data.py": "\\"\\"\\"Explicit class order; identical files cannot cross dataset splits.\\"\\"\\"\\n\\nfrom hashlib import sha256\\nfrom pathlib import Path\\n\\nfrom torch.utils.data import Dataset\\n\\nfrom glaucoma.pipeline import CLASSES, decode_image, preprocessing\\n\\n\\nclass FundusDataset(Dataset):\\n    def __init__(self, root):\\n        self.samples = []\\n        self.transform = preprocessing()\\n        for label, name in enumerate(CLASSES):\\n            folder = Path(root) / name\\n            paths = sorted(p for p in folder.rglob(\\"*\\")\\n                           if p.is_file() and p.suffix.lower() in {\\".png\\", \\".jpg\\", \\".jpeg\\"})\\n            if not paths:\\n                raise ValueError(f\\"No PNG/JPEG images found in {folder}.\\")\\n            self.samples.extend((p, label) for p in paths)\\n\\n    def __len__(self):\\n        return len(self.samples)\\n\\n    def __getitem__(self, index):\\n        path, label = self.samples[index]\\n        return self.transform(decode_image(path.read_bytes(), path.name)), label\\n\\n\\ndef check_split_overlap(*datasets):\\n    seen = {}\\n    for split, dataset in enumerate(datasets):\\n        for path, _ in dataset.samples:\\n            digest = sha256(path.read_bytes()).hexdigest()\\n            if digest in seen and seen[digest][0] != split:\\n                raise ValueError(f\\"Duplicate image crosses dataset splits: {path} and {seen[digest][1]}\\")\\n            seen[digest] = (split, path)\\n", "glaucoma/download.py": "\\"\\"\\"Fetch a published checkpoint and verify its exact SHA-256 before loading.\\"\\"\\"\\n\\nfrom hashlib import sha256\\nfrom pathlib import Path\\nimport urllib.request\\n\\n\\ndef download_checkpoint(url: str, expected_sha256: str, destination=\\"models/best_model.pth\\"):\\n    if not url.startswith(\\"https://github.com/Cxsmxnaut/GlaucomAI/releases/download/\\"):\\n        raise ValueError(\\"Only this project\'s HTTPS release URLs are supported.\\")\\n    if len(expected_sha256) != 64 or any(c not in \\"0123456789abcdef\\" for c in expected_sha256):\\n        raise ValueError(\\"Expected a lowercase SHA-256 checksum.\\")\\n    destination = Path(destination)\\n    if destination.exists():\\n        if sha256(destination.read_bytes()).hexdigest() == expected_sha256:\\n            return destination\\n        raise ValueError(\\"A different checkpoint already exists. Choose a new path or remove it explicitly.\\")\\n    destination.parent.mkdir(parents=True, exist_ok=True)\\n    temporary = destination.with_suffix(destination.suffix + \\".part\\")\\n    digest = sha256()\\n    try:\\n        request = urllib.request.Request(url, headers={\\"User-Agent\\": \\"GlaucomAI-research-demo\\"})\\n        total = 0\\n        with urllib.request.urlopen(request, timeout=60) as response, temporary.open(\\"wb\\") as target:\\n            while chunk := response.read(1024 * 1024):\\n                total += len(chunk)\\n                if total > 100 * 1024 * 1024:\\n                    raise ValueError(\\"Unexpected checkpoint size. Download stopped.\\")\\n                digest.update(chunk)\\n                target.write(chunk)\\n        if digest.hexdigest() != expected_sha256:\\n            raise ValueError(\\"Checkpoint checksum mismatch. The downloaded file was discarded.\\")\\n        temporary.replace(destination)\\n    finally:\\n        temporary.unlink(missing_ok=True)\\n    return destination\\n", "glaucoma/metrics.py": "\\"\\"\\"Small, auditable metrics without an additional ML dependency.\\"\\"\\"\\n\\nimport math\\n\\n\\ndef binary_metrics(labels, probabilities, threshold=0.5):\\n    if len(labels) != len(probabilities) or not labels:\\n        raise ValueError(\\"Provide equally sized, non-empty labels and scores.\\")\\n    if not 0 < threshold < 1:\\n        raise ValueError(\\"Threshold must be between zero and one.\\")\\n    if any(y not in (0, 1) for y in labels):\\n        raise ValueError(\\"Labels must be binary.\\")\\n    if any(not math.isfinite(p) or not 0 <= p <= 1 for p in probabilities):\\n        raise ValueError(\\"Scores must be finite probabilities.\\")\\n    tp = sum(y == 1 and p >= threshold for y, p in zip(labels, probabilities))\\n    tn = sum(y == 0 and p < threshold for y, p in zip(labels, probabilities))\\n    fp = sum(y == 0 and p >= threshold for y, p in zip(labels, probabilities))\\n    fn = sum(y == 1 and p < threshold for y, p in zip(labels, probabilities))\\n    return {\\n        \\"samples\\": len(labels), \\"true_positive\\": tp, \\"true_negative\\": tn,\\n        \\"false_positive\\": fp, \\"false_negative\\": fn,\\n        \\"sensitivity\\": tp / (tp + fn) if tp + fn else None,\\n        \\"specificity\\": tn / (tn + fp) if tn + fp else None,\\n        \\"accuracy\\": (tp + tn) / len(labels),\\n    }\\n\\n\\ndef select_threshold(labels, probabilities, target=0.88):\\n    \\"\\"\\"Choose on validation only; maximize specificity subject to target recall.\\"\\"\\"\\n    if set(labels) != {0, 1} or not 0 < target <= 1:\\n        raise ValueError(\\"Validation needs both classes and a target in (0, 1].\\")\\n    binary_metrics(labels, probabilities)\\n    candidates = sorted({0.001, *[p for p in probabilities if 0 < p < 1]})\\n    feasible = [t for t in candidates\\n                if binary_metrics(labels, probabilities, t)[\\"sensitivity\\"] >= target]\\n    if not feasible:\\n        raise ValueError(\\"No usable threshold meets the validation recall target.\\")\\n    return max(feasible, key=lambda t: (binary_metrics(labels, probabilities, t)[\\"specificity\\"], t))\\n", "glaucoma/notebook_ui.py": "\\"\\"\\"An in-notebook judging dashboard. No tunnel, web server or disk uploads.\\"\\"\\"\\n\\nimport base64\\nfrom html import escape\\nfrom io import BytesIO\\n\\nfrom IPython.display import HTML, display\\nimport ipywidgets as widgets\\nimport torch\\n\\nfrom glaucoma.pipeline import InputError, ModelError, decode_image, infer, load_model\\n\\nSTYLE = \\"\\"\\"\\n<style>\\n.gai {font-family:Inter,Arial,sans-serif;color:#193e43;line-height:1.55;max-width:1100px;background:#fff;padding:18px;border-radius:16px}\\n.gai * {box-sizing:border-box}\\n.gai .hero {background:#113f40;color:#fff;padding:30px 34px;border-radius:18px;margin:0 0 20px}\\n.gai .eyebrow {font-size:11px;letter-spacing:2px;text-transform:uppercase;color:#a4ded0}\\n.gai h1 {font-size:32px;letter-spacing:-1px;margin:8px 0;color:#fff;font-weight:650}\\n.gai h2 {font-size:21px;margin:7px 0 12px;color:#193e43}\\n.gai p {margin:7px 0}\\n.gai .sub {color:#c9dfdc;font-size:14px}\\n.gai .grid {display:grid;grid-template-columns:1fr 1fr;gap:20px;margin:16px 0}\\n.gai .card {background:#f6f9fa;border:1px solid #dce8e6;border-radius:14px;padding:24px}\\n.gai .image {background:#101e26;border-radius:10px;min-height:260px;display:flex;align-items:center;justify-content:center;overflow:hidden}\\n.gai img {display:block;max-height:340px;max-width:100%;object-fit:contain}\\n.gai .label {font-size:11px;letter-spacing:1.4px;text-transform:uppercase;color:#607c80}\\n.gai .score {font-size:48px;font-weight:650;letter-spacing:-2px;color:#147d70}\\n.gai .track {height:14px;border-radius:20px;background:#e0ebe9;overflow:hidden;margin:10px 0}\\n.gai .fill {height:100%;border-radius:20px;background:#147d70}\\n.gai .row {display:flex;justify-content:space-between;gap:16px;font-size:13px;margin:8px 0}\\n.gai .note {border-left:3px solid #c99341;padding:12px 16px;background:#fff8ec;color:#6e542d;font-size:12px;border-radius:4px;margin:16px 0}\\n.gai .muted {font-size:12px;color:#607c80}\\n.gai .pill {display:inline-block;padding:4px 10px;background:#e2efeb;border-radius:20px;color:#266f62;font-size:11px}\\n@media(max-width:700px){.gai .grid{grid-template-columns:1fr}.gai .hero{padding:22px}.gai h1{font-size:27px}}\\n</style>\\n\\"\\"\\"\\n\\nDISCLAIMER = \\"\\"\\"<div class=\\"note\\"><b>Research prototype \\u00b7 not a diagnosis.</b>\\nThis model has not been clinically validated. Scores can be wrong and are not calibrated probabilities\\nof disease. Non-referable does not mean healthy. A qualified eye-care professional must interpret\\nretinal findings. The app does not check whether an upload is a fundus photograph.</div>\\"\\"\\"\\n\\n\\ndef uploaded_file(value):\\n    \\"\\"\\"Normalize both Colab\'s ipywidgets 7 mapping and ipywidgets 8 sequence.\\"\\"\\"\\n    if not value:\\n        return None\\n    if isinstance(value, dict):\\n        name, file = next(iter(value.items()))\\n        return name, bytes(file[\\"content\\"])\\n    file = value[0]\\n    return file[\\"name\\"], bytes(file[\\"content\\"])\\n\\n\\ndef image_data_url(image):\\n    preview = image.copy()\\n    preview.thumbnail((800, 800))\\n    buffer = BytesIO()\\n    preview.save(buffer, format=\\"PNG\\")\\n    return \\"data:image/png;base64,\\" + base64.b64encode(buffer.getvalue()).decode()\\n\\n\\ndef result_html(image, result):\\n    score = result[\\"referable\\"] * 100\\n    decision = \\"Above the referral-score threshold\\" if result[\\"flagged\\"] else \\"Below the referral-score threshold\\"\\n    return STYLE + f\\"\\"\\"<div class=\\"gai\\"><div class=\\"grid\\">\\n<div class=\\"card\\"><div class=\\"label\\">01 / Uploaded fundus image</div>\\n<div class=\\"image\\"><img src=\\"{image_data_url(image)}\\" alt=\\"Uploaded image for research analysis\\"/></div>\\n<p class=\\"muted\\">{image.width} \\u00d7 {image.height} original \\u00b7 224 \\u00d7 224 model input</p></div>\\n<div class=\\"card\\"><div class=\\"label\\">02 / Model output</div><h2>{decision}</h2>\\n<div class=\\"score\\">{score:.1f}<span style=\\"font-size:24px\\">%</span></div>\\n<p>Referable glaucoma model score</p>\\n<div class=\\"track\\" role=\\"progressbar\\" aria-label=\\"Referable glaucoma score\\" aria-valuemin=\\"0\\" aria-valuemax=\\"100\\" aria-valuenow=\\"{score:.1f}\\"><div class=\\"fill\\" style=\\"width:{score:.2f}%\\"></div></div>\\n<div class=\\"row\\"><span>Non-referable score</span><b>{result[\'non_referable\']:.1%}</b></div>\\n<div class=\\"row\\"><span>Referral-score threshold</span><b>{result[\'threshold\']:.1%}</b></div>\\n<div class=\\"row\\"><span>CPU preprocessing + inference</span><b>{result[\'elapsed_ms\']:.0f} ms</b></div>\\n<p class=\\"muted\\">The threshold was selected on validation images. This is a research routing signal, not a treatment recommendation.</p>\\n</div></div>{DISCLAIMER}</div>\\"\\"\\"\\n\\n\\ndef display_assistant(model_path=\\"models/best_model.pth\\"):\\n    \\"\\"\\"Call once in Colab/Jupyter after training; session memory only.\\"\\"\\"\\n    torch.set_num_threads(2)\\n    loaded = load_model(model_path)\\n    display(HTML(STYLE + f\\"\\"\\"<div class=\\"gai\\"><div class=\\"hero\\">\\n<div class=\\"eyebrow\\">GlaucomAI / AI + Health</div>\\n<h1>A clearer view. An earlier signal.</h1>\\n<p class=\\"sub\\">Explore glaucoma screening with a retinal photograph and a trained EfficientNet-B0.</p>\\n<span class=\\"pill\\">Trained checkpoint connected \\u00b7 CPU inference</span></div>\\n<p><b>Upload a fundus photograph</b> <span class=\\"muted\\">PNG or JPEG \\u00b7 under 10 MB \\u00b7 one image at a time</span></p>\\n{DISCLAIMER}</div>\\"\\"\\"))\\n    uploader = widgets.FileUpload(accept=\\".png,.jpg,.jpeg\\", multiple=False, description=\\"Choose image\\")\\n    analyze = widgets.Button(description=\\"Analyze image \\u2192\\", button_style=\\"success\\", disabled=True,\\n                             layout=widgets.Layout(width=\\"180px\\"))\\n    reset = widgets.Button(description=\\"Clear image\\", icon=\\"trash\\")\\n    output = widgets.Output()\\n    state = {\\"image\\": None}\\n\\n    def clear_image():\\n        if state[\\"image\\"] is not None:\\n            state[\\"image\\"].close()\\n        state[\\"image\\"] = None\\n        analyze.disabled = True\\n        output.clear_output()\\n\\n    def uploaded(change):\\n        clear_image()\\n        if not uploader.value:\\n            return\\n        try:\\n            name, content = uploaded_file(uploader.value)\\n            state[\\"image\\"] = decode_image(content, name)\\n            analyze.disabled = False\\n            with output:\\n                display(HTML(STYLE + f\'<div class=\\"gai\\"><p class=\\"pill\\">Image decoded \\u00b7 ready to analyze</p><div class=\\"image\\"><img src=\\"{image_data_url(state[\\"image\\"])}\\" alt=\\"Uploaded image preview\\"/></div></div>\'))\\n        except InputError as exc:\\n            with output:\\n                display(HTML(f\'<div class=\\"gai\\"><div class=\\"note\\">{escape(str(exc))}</div></div>\'))\\n\\n    def run_analysis(button):\\n        if state[\\"image\\"] is None:\\n            return\\n        analyze.disabled = True\\n        analyze.description = \\"Analyzing\\u2026\\"\\n        output.clear_output(wait=True)\\n        try:\\n            result = infer(state[\\"image\\"], loaded)\\n            with output:\\n                display(HTML(result_html(state[\\"image\\"], result)))\\n        except (ModelError, RuntimeError) as exc:\\n            with output:\\n                display(HTML(f\'<div class=\\"gai\\"><div class=\\"note\\">Analysis unavailable: {escape(str(exc))}</div></div>\'))\\n        finally:\\n            analyze.description = \\"Analyze image \\u2192\\"\\n            analyze.disabled = state[\\"image\\"] is None\\n\\n    def reset_upload(button):\\n        clear_image()\\n        if isinstance(uploader.value, dict):\\n            # ipywidgets 7 computes its read-only value from these synced traits.\\n            uploader.metadata = []\\n            uploader.data = []\\n            uploader._counter = 0\\n            uploader.set_trait(\\"value\\", {})\\n        else:\\n            uploader.value = ()\\n\\n    uploader.observe(uploaded, names=\\"value\\")\\n    analyze.on_click(run_analysis)\\n    reset.on_click(reset_upload)\\n    display(widgets.VBox([widgets.HBox([uploader, analyze, reset]), output]))\\n    return {\\"uploader\\": uploader, \\"analyze\\": analyze, \\"reset\\": reset, \\"output\\": output}\\n", "glaucoma/pipeline.py": "\\"\\"\\"Shared, deterministic preprocessing and CPU inference. Never stores images.\\"\\"\\"\\n\\nfrom dataclasses import dataclass\\nfrom io import BytesIO\\nfrom pathlib import Path\\nimport math\\nimport time\\nimport warnings\\n\\nfrom PIL import Image, ImageOps, UnidentifiedImageError\\nimport torch\\nfrom torch import nn\\nfrom torchvision import models, transforms\\n\\nMAX_BYTES = 10 * 1024 * 1024\\nMAX_PIXELS = 25_000_000\\nCLASSES = [\\"non_referable\\", \\"referable\\"]\\nMEAN = [0.485, 0.456, 0.406]\\nSTD = [0.229, 0.224, 0.225]\\nPREPROCESSING = {\\"size\\": [224, 224], \\"mean\\": MEAN, \\"std\\": STD}\\n\\n\\nclass InputError(ValueError):\\n    pass\\n\\n\\nclass ModelError(ValueError):\\n    pass\\n\\n\\ndef decode_image(data: bytes, filename: str) -> Image.Image:\\n    \\"\\"\\"Check the actual format, byte size and pixel count, then remove metadata.\\"\\"\\"\\n    if Path(filename).suffix.lower() not in {\\".png\\", \\".jpg\\", \\".jpeg\\"}:\\n        raise InputError(\\"Choose a PNG or JPEG image.\\")\\n    if not data or len(data) >= MAX_BYTES:\\n        raise InputError(\\"Choose a non-empty image smaller than 10 MB.\\")\\n    try:\\n        with warnings.catch_warnings():\\n            warnings.simplefilter(\\"error\\", Image.DecompressionBombWarning)\\n            with Image.open(BytesIO(data)) as source:\\n                if source.format not in {\\"PNG\\", \\"JPEG\\"}:\\n                    raise InputError(\\"The file contents must be PNG or JPEG.\\")\\n                if source.width * source.height > MAX_PIXELS:\\n                    raise InputError(\\"The image exceeds the 25-megapixel safety limit.\\")\\n                source.verify()\\n            with Image.open(BytesIO(data)) as source:\\n                oriented = ImageOps.exif_transpose(source).convert(\\"RGB\\")\\n                # A new image avoids retaining EXIF, names or other source metadata.\\n                return Image.frombytes(\\"RGB\\", oriented.size, oriented.tobytes())\\n    except InputError:\\n        raise\\n    except (UnidentifiedImageError, OSError, SyntaxError, ValueError,\\n            Image.DecompressionBombError, Image.DecompressionBombWarning) as exc:\\n        raise InputError(\\"This image could not be decoded. Try another PNG or JPEG.\\") from exc\\n\\n\\ndef preprocessing():\\n    return transforms.Compose([\\n        transforms.Resize((224, 224), antialias=True),\\n        transforms.ToTensor(),\\n        transforms.Normalize(MEAN, STD),\\n    ])\\n\\n\\ndef build_model(pretrained: bool = False) -> nn.Module:\\n    weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None\\n    model = models.efficientnet_b0(weights=weights)\\n    model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)\\n    return model\\n\\n\\n@dataclass\\nclass LoadedModel:\\n    model: nn.Module\\n    threshold: float\\n    metadata: dict\\n\\n\\ndef load_model(path: str | Path) -> LoadedModel:\\n    \\"\\"\\"Load only a local, trusted checkpoint. Never download inference weights.\\"\\"\\"\\n    try:\\n        checkpoint = torch.load(path, map_location=\\"cpu\\", weights_only=True)\\n        if not isinstance(checkpoint, dict):\\n            raise ModelError(\\"Checkpoint must be a dictionary exported by train.py.\\")\\n        if checkpoint.get(\\"architecture\\") != \\"efficientnet_b0\\":\\n            raise ModelError(\\"Checkpoint architecture must be efficientnet_b0.\\")\\n        if checkpoint.get(\\"classes\\") != CLASSES:\\n            raise ModelError(\\"Checkpoint class order must be non_referable=0, referable=1.\\")\\n        if checkpoint.get(\\"preprocessing\\") != PREPROCESSING:\\n            raise ModelError(\\"Checkpoint preprocessing does not match this app.\\")\\n        threshold = float(checkpoint[\\"threshold\\"])\\n        if not math.isfinite(threshold) or not 0 < threshold < 1:\\n            raise ModelError(\\"Checkpoint threshold must be between zero and one.\\")\\n        model = build_model()\\n        model.load_state_dict(checkpoint[\\"state_dict\\"], strict=True)\\n        model.eval()\\n        if any(not torch.isfinite(p).all() for p in model.state_dict().values()):\\n            raise ModelError(\\"Checkpoint contains invalid model values.\\")\\n        return LoadedModel(model, threshold, {\\n            key: value for key, value in checkpoint.items() if key != \\"state_dict\\"\\n        })\\n    except ModelError:\\n        raise\\n    except Exception as exc:\\n        raise ModelError(\\"Cannot load this checkpoint. Export a compatible model with train.py.\\") from exc\\n\\n\\ndef infer(image: Image.Image, loaded: LoadedModel) -> dict:\\n    start = time.perf_counter()\\n    with torch.inference_mode():\\n        tensor = preprocessing()(image.convert(\\"RGB\\")).unsqueeze(0)\\n        probabilities = torch.softmax(loaded.model(tensor), dim=1)[0]\\n    values = probabilities.tolist()\\n    if len(values) != 2 or not all(math.isfinite(p) for p in values):\\n        raise ModelError(\\"The model returned invalid scores. No result is available.\\")\\n    return {\\n        \\"non_referable\\": values[0],\\n        \\"referable\\": values[1],\\n        \\"threshold\\": loaded.threshold,\\n        \\"flagged\\": values[1] >= loaded.threshold,\\n        \\"elapsed_ms\\": (time.perf_counter() - start) * 1000,\\n        \\"demo\\": False,\\n    }\\n", "glaucoma/runtime.py": "\\"\\"\\"Stream child-process logs through notebook stdout and preserve failures.\\"\\"\\"\\n\\nimport subprocess\\n\\n\\ndef run_logged(arguments):\\n    with subprocess.Popen(arguments, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,\\n                          text=True, bufsize=1) as process:\\n        for line in process.stdout:\\n            print(line, end=\\"\\", flush=True)\\n        status = process.wait()\\n    if status:\\n        raise subprocess.CalledProcessError(status, arguments)\\n", "prepare_data.py": "\\"\\"\\"Download the pinned Kaggle dataset and preserve its supplied splits.\\"\\"\\"\\n\\nimport argparse\\nfrom collections import Counter\\nimport csv\\nfrom hashlib import sha256\\nimport json\\nfrom pathlib import Path\\nimport shutil\\n\\nfrom glaucoma.data import FundusDataset, check_split_overlap\\nfrom glaucoma.pipeline import decode_image\\n\\nDATASET = \\"deathtrooper/glaucoma-dataset-eyepacs-airogs-light-v2/versions/4\\"\\nSOURCE = \\"https://www.kaggle.com/datasets/deathtrooper/glaucoma-dataset-eyepacs-airogs-light-v2/versions/4\\"\\nSPLITS = {\\"train\\": \\"train\\", \\"validation\\": \\"val\\", \\"test\\": \\"test\\"}\\nLABELS = {\\"NRG\\": \\"non_referable\\", \\"RG\\": \\"referable\\"}\\n\\n\\ndef prepare(source: Path, output: Path):\\n    \\"\\"\\"Validate every file before copying; no guessed labels or random re-split.\\"\\"\\"\\n    roots = [p for p in source.rglob(\\"*\\") if p.is_dir()\\n             and all((p / name).is_dir() for name in SPLITS)]\\n    if all((source / name).is_dir() for name in SPLITS):\\n        roots.insert(0, source)\\n    if len(roots) != 1:\\n        raise ValueError(\\"Expected one dataset root with train/, validation/ and test/ folders.\\")\\n    root = roots[0]\\n    if output.exists():\\n        raise ValueError(f\\"{output} already exists. Choose a new output directory to avoid mixing splits.\\")\\n    rows = []\\n    seen = {}\\n    for source_split, split in SPLITS.items():\\n        for source_class, label in LABELS.items():\\n            folder = root / source_split / source_class\\n            paths = sorted(p for p in folder.rglob(\\"*\\")\\n                           if p.is_file() and p.suffix.lower() in {\\".jpg\\", \\".jpeg\\", \\".png\\"})\\n            if not paths:\\n                raise ValueError(f\\"No images found in {folder}.\\")\\n            for path in paths:\\n                raw = path.read_bytes()\\n                decode_image(raw, path.name).close()\\n                digest = sha256(raw).hexdigest()\\n                if digest in seen:\\n                    raise ValueError(f\\"Duplicate image: {path} and {seen[digest]}. Review source data.\\")\\n                seen[digest] = path\\n                rows.append({\\n                    \\"source\\": path.relative_to(root).as_posix(), \\"split\\": split,\\n                    \\"label\\": label, \\"sha256\\": digest,\\n                    \\"destination\\": f\\"{split}/{label}/{digest[:16]}{path.suffix.lower()}\\",\\n                })\\n    # Validation succeeds before creating the output directory.\\n    output.mkdir(parents=True)\\n    for row in rows:\\n        destination = output / row[\\"destination\\"]\\n        destination.parent.mkdir(parents=True, exist_ok=True)\\n        shutil.copyfile(root / row[\\"source\\"], destination)\\n    with (output / \\"manifest.csv\\").open(\\"w\\", newline=\\"\\") as stream:\\n        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))\\n        writer.writeheader()\\n        writer.writerows(rows)\\n    check_split_overlap(*[FundusDataset(output / split) for split in SPLITS.values()])\\n    provenance = {\\n        \\"dataset\\": DATASET, \\"source_url\\": SOURCE,\\n        \\"counts\\": dict(Counter(f\\"{r[\'split\']}/{r[\'label\']}\\" for r in rows)),\\n        \\"split_policy\\": \\"Preserve the dataset publisher\'s train/validation/test splits.\\",\\n        \\"patient_separation\\": \\"Not independently verified. File-hash disjointness does not establish patient disjointness.\\",\\n        \\"label_mapping\\": LABELS,\\n    }\\n    (output / \\"provenance.json\\").write_text(json.dumps(provenance, indent=2))\\n    return provenance\\n\\n\\ndef main():\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\\"--source\\", type=Path, help=\\"Use an already downloaded Kaggle dataset root.\\")\\n    parser.add_argument(\\"--output\\", type=Path, default=Path(\\"data/prepared\\"))\\n    args = parser.parse_args()\\n    if args.source:\\n        source = args.source\\n    else:\\n        import kagglehub\\n        source = Path(kagglehub.dataset_download(DATASET))\\n    print(json.dumps(prepare(source, args.output), indent=2))\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "train.py": "\\"\\"\\"Train a frozen EfficientNet-B0 backbone, select threshold on validation only.\\"\\"\\"\\n\\nimport argparse\\nimport json\\nfrom pathlib import Path\\nimport random\\n\\nimport torch\\nfrom torch import nn\\nfrom torch.utils.data import DataLoader, TensorDataset\\n\\nfrom glaucoma.data import FundusDataset, check_split_overlap\\nfrom glaucoma.metrics import binary_metrics, select_threshold\\nfrom glaucoma.pipeline import CLASSES, PREPROCESSING, build_model\\n\\n\\ndef extract_features(model, dataset, device, batch_size):\\n    \\"\\"\\"A frozen backbone with deterministic preprocessing only needs one pass.\\"\\"\\"\\n    model.eval()\\n    features, targets = [], []\\n    loader = DataLoader(dataset, batch_size=batch_size)\\n    with torch.inference_mode():\\n        for index, (images, labels) in enumerate(loader):\\n            output = model.features(images.to(device))\\n            output = model.avgpool(output).flatten(1)\\n            features.append(output.cpu())\\n            targets.append(labels)\\n            if (index + 1) % 25 == 0 or index + 1 == len(loader):\\n                print(f\\"  Backbone features: {min((index + 1) * batch_size, len(dataset))}/{len(dataset)}\\", flush=True)\\n    # Clone outside inference mode so the classifier can save inputs for backward.\\n    return TensorDataset(torch.cat(features).clone(), torch.cat(targets).clone())\\n\\n\\ndef main():\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\\"--data\\", type=Path, default=Path(\\"data/prepared\\"))\\n    parser.add_argument(\\"--output\\", type=Path, default=Path(\\"models/best_model.pth\\"))\\n    parser.add_argument(\\"--epochs\\", type=int, default=12)\\n    parser.add_argument(\\"--batch-size\\", type=int, default=16)\\n    parser.add_argument(\\"--seed\\", type=int, default=42)\\n    parser.add_argument(\\"--device\\", choices=[\\"auto\\", \\"cpu\\", \\"cuda\\", \\"mps\\"], default=\\"auto\\")\\n    args = parser.parse_args()\\n    if args.epochs < 1 or args.batch_size < 1:\\n        parser.error(\\"epochs and batch-size must be positive\\")\\n    torch.manual_seed(args.seed)\\n    random.seed(args.seed)\\n    torch.set_num_threads(2)\\n    train = FundusDataset(args.data / \\"train\\")\\n    val = FundusDataset(args.data / \\"val\\")\\n    check_split_overlap(train, val)\\n    device = torch.device((\\"cuda\\" if torch.cuda.is_available() else \\"cpu\\")\\n                          if args.device == \\"auto\\" else args.device)\\n    print(f\\"Training on {device}; {len(train)} train / {len(val)} validation images.\\")\\n    print(\\"ImageNet weights download on first training run. No dataset is downloaded.\\")\\n    model = build_model(pretrained=True).to(device)\\n    for p in model.features.parameters():\\n        p.requires_grad = False\\n    optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=1e-3)\\n    criterion = nn.CrossEntropyLoss()\\n    print(\\"Extracting frozen backbone features once for train and validation.\\", flush=True)\\n    train_features = extract_features(model, train, device, args.batch_size)\\n    val_features = extract_features(model, val, device, args.batch_size)\\n    loader = DataLoader(train_features, batch_size=args.batch_size, shuffle=True)\\n    val_loader = DataLoader(val_features, batch_size=args.batch_size)\\n    best_loss = float(\\"inf\\")\\n    history = []\\n    args.output.parent.mkdir(parents=True, exist_ok=True)\\n    for epoch in range(args.epochs):\\n        model.train()\\n        model.features.eval()  # Also freeze batchnorm statistics in the backbone.\\n        train_loss = 0.0\\n        for features, labels in loader:\\n            features, labels = features.to(device), labels.to(device)\\n            optimizer.zero_grad()\\n            loss = criterion(model.classifier(features), labels)\\n            loss.backward()\\n            optimizer.step()\\n            train_loss += loss.item() * len(labels)\\n        model.eval()\\n        val_loss, all_labels, probabilities = 0.0, [], []\\n        with torch.inference_mode():\\n            for features, labels in val_loader:\\n                logits = model.classifier(features.to(device))\\n                val_loss += criterion(logits, labels.to(device)).item() * len(labels)\\n                all_labels.extend(labels.tolist())\\n                probabilities.extend(logits.softmax(dim=1)[:, 1].cpu().tolist())\\n        val_loss /= len(val)\\n        print(f\\"Epoch {epoch + 1:02d}: train loss {train_loss / len(train):.4f}, val loss {val_loss:.4f}\\")\\n        history.append({\\"epoch\\": epoch + 1, \\"train_loss\\": train_loss / len(train), \\"validation_loss\\": val_loss})\\n        args.output.with_suffix(\\".history.json\\").write_text(json.dumps(history, indent=2))\\n        if val_loss < best_loss:\\n            best_loss = val_loss\\n            threshold = select_threshold(all_labels, probabilities)\\n            torch.save({\\n                \\"architecture\\": \\"efficientnet_b0\\", \\"classes\\": CLASSES,\\n                \\"preprocessing\\": PREPROCESSING, \\"threshold\\": threshold,\\n                \\"state_dict\\": {k: v.detach().cpu() for k, v in model.state_dict().items()},\\n                \\"epoch\\": epoch + 1, \\"seed\\": args.seed,\\n                \\"training_samples\\": len(train),\\n                \\"dataset_provenance\\": json.loads((args.data / \\"provenance.json\\").read_text())\\n                if (args.data / \\"provenance.json\\").exists() else {\\"source\\": \\"user_supplied\\"},\\n                \\"validation_metrics\\": binary_metrics(all_labels, probabilities, threshold),\\n                \\"validation_loss\\": val_loss,\\n                \\"status\\": \\"research_only_unvalidated\\",\\n            }, args.output)\\n    print(f\\"Saved {args.output}. Run evaluate.py on the held-out test split.\\")\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n"}')
for name, content in SOURCES.items():
    destination = PROJECT / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(content)
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
print('Project ready:', PROJECT)

## 3. Download the Kaggle dataset
We use **EyePACS-AIROGS-light-V2, version 4**. Its publisher supplies train, validation, and test folders.
We preserve those splits and map `NRG → non_referable (0)` and `RG → referable (1)` explicitly.
This is a balanced subset, so its class mix does not represent disease prevalence in the general population.

Public downloads may work without signing in. If Kaggle requests authentication, add a Colab secret
named `KAGGLE_API_TOKEN` using the key icon in the sidebar, grant this notebook access, then rerun this cell.
You can create the token in [Kaggle settings](https://www.kaggle.com/settings/api).
Read the dataset page and its linked source terms before use or redistribution.

In [ ]:
import kagglehub
from prepare_data import DATASET
raw_path = Path(kagglehub.dataset_download(DATASET))
print("Downloaded dataset:", DATASET)
print("Dataset root:", raw_path)

## 4. Validate images and preserve the splits
This checks file contents, rejects broken or oversized images, and checks for byte-identical duplicates.
It writes a manifest with source paths and hashes. File-hash checks cannot prove that different photographs
come from different patients; the publisher's patient separation has not been independently verified.
The prepared data stays in the temporary runtime and is not uploaded to GitHub.

In [ ]:
from prepare_data import prepare
data_path = PROJECT / "data/prepared"
if data_path.exists():
    raise RuntimeError("Prepared data already exists. To reuse it, skip this cell. For a fresh run, delete the runtime first.")
provenance = prepare(raw_path, data_path)
print(json.dumps(provenance, indent=2))

## 5. Train EfficientNet-B0
We start from ImageNet weights, freeze the feature extractor, and train a two-class head for 12 epochs.
Every image is resized to **224 × 224**, converted to RGB, and normalized using ImageNet mean and standard deviation.
Frozen batch-normalization statistics stay frozen, too.
Because preprocessing is deterministic and the backbone is frozen, its features are extracted once
into memory. The classification head then trains on those features for all 12 epochs.

The best checkpoint is chosen using **validation loss**. Its routing threshold is chosen on validation
images to target at least 88% recall while maximizing specificity. Test images do not choose weights or thresholds.
Meeting the validation target does **not** mean the model will meet it on the test set or on new patients.

In [ ]:
from glaucoma.runtime import run_logged
EPOCHS = 12
BATCH_SIZE = 32
run_logged([sys.executable, "-u", "train.py", "--data", str(data_path),
                "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE),
                "--output", "models/best_model.pth"])

## 6. Inspect training
A falling training loss with rising validation loss can be a sign of overfitting:
the model is learning the training examples more than useful patterns.

In [ ]:
import matplotlib.pyplot as plt
history = json.loads(Path("models/best_model.history.json").read_text())
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot([r["epoch"] for r in history], [r["train_loss"] for r in history], label="Training", color="#147d70")
ax.plot([r["epoch"] for r in history], [r["validation_loss"] for r in history], label="Validation", color="#c99341")
ax.set(xlabel="Epoch", ylabel="Cross-entropy loss", title="Training and validation loss")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## 7. Evaluate once on the held-out test set
This uses the saved threshold without tuning it on test images. The report includes sensitivity
(how many referable images it catches), specificity (how many non-referable images it leaves unflagged),
and false positives/negatives. We also time the real CPU pipeline after one warmup.
The 2-second target is measured for decoding, preprocessing, and inference; browser transfer and rendering
need a separate end-to-end check. CPU performance here describes this Colab machine, not every laptop.

In [ ]:
run_logged([sys.executable, "-u", "evaluate.py", "--data", str(data_path),
            "--model", "models/best_model.pth", "--output", "reports/evaluation.json"])
report = json.loads(Path("reports/evaluation.json").read_text())
metrics = report["metrics"]
print(f"TEST sensitivity: {metrics['sensitivity']:.1%} | specificity: {metrics['specificity']:.1%}")
print(f"CPU pipeline p95: {report['p95_pipeline_ms']:.0f} ms")
print("88% sensitivity target:", "MET on this test set" if report["sensitivity_target_met"] else "NOT MET")
print("2-second pipeline target:", "MET on this CPU" if report["latency_target_met"] else "NOT MET")
matrix = [[metrics["true_negative"], metrics["false_positive"]],
          [metrics["false_negative"], metrics["true_positive"]]]
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(matrix, cmap="Greens", vmin=0)
for y in range(2):
    for x in range(2):
        ax.text(x, y, str(matrix[y][x]), ha="center", va="center", fontsize=20,
                color="white" if matrix[y][x] > max(map(max, matrix)) / 2 else "#193e43")
ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=["Non-referable", "Referable"],
       yticklabels=["Non-referable", "Referable"], xlabel="Predicted", ylabel="Dataset label",
       title="Held-out test confusion matrix")
plt.tight_layout()
plt.show()

## 8. Try the trained model
Upload one PNG/JPEG under 10 MB, then select **Analyze image**. This uses the trained checkpoint;
it does not generate sample scores. It runs on CPU, even if training used a GPU.
Format validation does not detect whether an image is a retina or whether it is suitable for diagnosis.

Uploads are held in this runtime's memory. Use **Clear image** when done. Clear cell outputs before
sharing the notebook, and keep **Edit → Notebook settings → Omit code cell output when saving** enabled.
Notebook outputs may contain the uploaded image. Use de-identified research images for the demo.

In [ ]:
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass
from glaucoma.notebook_ui import display_assistant
dashboard = display_assistant("models/best_model.pth")

## 9. Export the trained model and evidence
Colab runtimes are temporary. Download this archive before disconnecting.
It includes the model, its metadata, training history, evaluation report, data provenance, and exact installed package versions.
It does **not** include the dataset, your API token, or uploaded demo images.
The repository contains code and a notebook; trained artifacts are separate downloads.

In [ ]:
import zipfile
archive = PROJECT / "GlaucomAI_trained_artifacts.zip"
versions = subprocess.run([sys.executable, "-m", "pip", "freeze"], check=True, capture_output=True, text=True).stdout
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    for name in ["models/best_model.pth", "models/best_model.history.json", "reports/evaluation.json",
                 "data/prepared/provenance.json"]:
        bundle.write(name, arcname=name)
    bundle.writestr("environment.txt", versions)
print("Artifacts ready:", archive.name)
from IPython.display import FileLink, display
display(FileLink(archive.name))

### Download to your computer (run this cell when ready)
Colab's download helper opens a browser download. Outside Colab, use the archive link above.

In [ ]:
# Uncomment these two lines in Colab when you want to download the archive:
# from google.colab import files
# files.download(str(archive))

## Reopen a trained model without retraining
In a fresh runtime, run steps 1–2. Upload your trusted `best_model.pth` with the Colab Files sidebar
into `/content/GlaucomAI/models/` (create the `models` folder if needed), then run step 8.
Only load checkpoints you trust. Do not rerun steps 3–7 just to show the demo.

## What this prototype proves—and what it does not
This is a binary image classifier. It does **not** segment the optic cup/disc, measure cup-to-disc ratio,
establish the cause of a prediction, or diagnose glaucoma. Softmax scores are not calibrated clinical risks.
Publisher splits and exact-duplicate checks are useful, but patient separation and cross-dataset performance
still need independent verification. The 88% recall and 2-second latency values are targets until measured.

**Sources:** [dataset and citations](https://www.kaggle.com/datasets/deathtrooper/glaucoma-dataset-eyepacs-airogs-light-v2/versions/4),
[KaggleHub authentication](https://github.com/Kaggle/kagglehub#authenticate),
[EfficientNet-B0](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.efficientnet_b0.html),
[Colab limits and sharing](https://research.google.com/colaboratory/faq.html).